# DeepLab Archaeology Segmentation Runner

This notebook only orchestrates reproducible scripts from `03_multiclass_segmentation_deeplab`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import torch

print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/MataNerdy/Geodata_Archaeology_CV.git')
BRANCH = os.environ.get('BRANCH', 'main')
REPO_DIR = Path('/kaggle/working/Geodata_Archaeology_CV')

if REPO_DIR.exists():
    print('Repo already exists. Pulling latest changes...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], cwd=REPO_DIR, check=True)
else:
    print('Cloning repo...')
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

SEG_DIR = REPO_DIR / '03_multiclass_segmentation_deeplab'
print('Using:', SEG_DIR)
os.environ['PYTHONPATH'] = str(SEG_DIR) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('PYTHONPATH prefix:', SEG_DIR)
subprocess.run(['git', 'log', '--oneline', '-3'], cwd=REPO_DIR, check=True)

In [ ]:
DATA_ROOT = Path(os.environ.get('DATA_ROOT', '/kaggle/input/datasets/matanerdy/kurgans-dataset/segmentation_dataset/segmentation_dataset'))
RUN_ROOT = Path(os.environ.get('RUN_ROOT', str(SEG_DIR / 'runs')))

print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT:', RUN_ROOT)
assert (DATA_ROOT / 'metadata.csv').exists(), f'metadata.csv not found: {DATA_ROOT}'
assert (DATA_ROOT / 'images').is_dir(), f'images dir not found: {DATA_ROOT / "images"}'
assert (DATA_ROOT / 'masks').is_dir(), f'masks dir not found: {DATA_ROOT / "masks"}'

## Smoke Test

Runs a tiny 2-epoch binary LiDAR DeepLab experiment.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/train.py',
    '--config', 'configs/binary_kurgan.yaml',
    '--data-root', str(DATA_ROOT),
    '--out-dir', str(RUN_ROOT / 'smoke_test'),
    '--epochs', '2',
    '--batch-size', '2',
    '--save-samples', '2',
], cwd=SEG_DIR, check=True)

## Selected Experiment

Default: first serious run is binary kurgan DeepLab / Li only. Override `EXPERIMENT_CONFIG` if needed.

In [ ]:
experiment_config = os.environ.get('EXPERIMENT_CONFIG', 'configs/binary_kurgan.yaml')
experiment_name = Path(experiment_config).stem
out_dir = RUN_ROOT / experiment_name

subprocess.run([
    sys.executable, 'scripts/train.py',
    '--config', experiment_config,
    '--data-root', str(DATA_ROOT),
    '--out-dir', str(out_dir),
], cwd=SEG_DIR, check=True)

subprocess.run([
    sys.executable, 'scripts/evaluate.py',
    '--checkpoint', str(out_dir / 'best_model.pth'),
    '--data-root', str(DATA_ROOT),
    '--out-dir', str(out_dir),
], cwd=SEG_DIR, check=True)

In [ ]:
import pandas as pd
from IPython.display import display, Image

history_path = out_dir / 'history.csv'
evaluation_path = out_dir / 'evaluation.csv'
if history_path.exists():
    display(pd.read_csv(history_path).tail())
if evaluation_path.exists():
    display(pd.read_csv(evaluation_path))
image_path = out_dir / 'prediction_examples.png'
if image_path.exists():
    display(Image(filename=str(image_path)))

In [ ]:
zip_path = Path('/kaggle/working/deeplab_runs.zip')
subprocess.run(['zip', '-r', str(zip_path), str(RUN_ROOT)], check=True)
print('Saved:', zip_path)